# NeuroMANCER Node and System classes and modules tutorial

This script demonstrates how to use NeuroMANCER Node to wrap arbitrary callable
into symbolic representation that can be used in NeuroMANCER problem formulation.


### Install (Colab only)
Skip this step when running locally.

In [ ]:
!pip install neuromancer

*Note: When running on Colab, one might encounter a pip dependency error with Lida 0.0.10. This can be ignored*

### Import

In [1]:
import torch
from neuromancer.system import Node, System

## Node

**Node** is a simple class to create symbolic modules out of arbitrary PyTorch callables.
Node class is wrapping the callable and defines the computational node based 
on input_keys and output_keys that define computational node connections through 
intermediate dictionaries. Complex symbolic architectures can be constructed by connecting
input and output keys of a set of Nodes via System and Problem classes.
   

In [2]:
# 1, wrap nn.Linear into Node
net_1 = torch.nn.Linear(1, 1)
node_1 = Node(net_1, ['x1'], ['y1'])
# print input and output keys
print(node_1.input_keys)
print(node_1.output_keys)
# evaluate forward pass of the node with dictionary input dataset
print(node_1({'x1': torch.rand(1)}))

['x1']
['y1']
{'y1': tensor([-0.3764], grad_fn=<ViewBackward0>)}


In [3]:
# 2, wrap nn.Sequential into Node
net_2 = torch.nn.Sequential(torch.nn.Linear(2, 5),
                            torch.nn.ReLU(),
                            torch.nn.Linear(5, 3),
                            torch.nn.ReLU(),
                            torch.nn.Linear(3, 1))
node_2 = Node(net_2, ['x2'], ['y2'])
print(node_2.input_keys)
print(node_2.output_keys)
# evaluate forward pass of the node with dictionary input dataset
print(node_2({'x2': torch.rand(2)}))

['x2']
['y2']
{'y2': tensor([-0.2679], grad_fn=<ViewBackward0>)}


In [4]:
# 3, wrap arbitrary callable into Node - allows for unwrapping the inputs
fun_1 = lambda x1, x2: 2.*x1 - x2**2
node_3 = Node(fun_1, ['y1', 'y2'], ['y3'], name='quadratic')
print(node_3.input_keys)
print(node_3.output_keys)
# evaluate forward pass of the node with dictionary input dataset
print(node_3({'y1': torch.rand(2), 'y2': torch.rand(2)}))

['y1', 'y2']
['y3']
{'y3': tensor([ 0.7573, -0.4376])}


In [14]:
# 4, wrap callable with multiple inputs and outputs
def fun_2(x1, x2):
    return x1**2, x2**2
node_4 = Node(fun_2, ['x1', 'x2'], ['x1^2', 'x2^2'], name='square')
print(node_4.input_keys)
print(node_4.output_keys)
# evaluate forward pass of the node with dictionary input dataset
print(node_4({'x1': torch.rand(2), 'x2': torch.rand(2)}))

['x1', 'x2']
['x1^2', 'x2^2']
{'x1^2': tensor([0.0222, 0.7594]), 'x2^2': tensor([0.1142, 0.1317])}


## Modules

NeuroMANCER also provides implementation of useful building blocks for
creating custom neural architectures. These include:
* modules.blocks          - neural architecures
* modules.activations     - custom activation functions    
* modules.functions       - useful callables 
* modules.gnn             - graph neural nets
* modules.rnn             - recurent neural nets
* modules.solvers         - iterative solvers for constrained optimization
* slim.linear             - linear algebra factorizations for weights
        
Next set of example shows how to wrap NeuroMANCER modules into Node

In [8]:
from neuromancer.modules import blocks
from neuromancer.modules import activations
from neuromancer import slim

In [9]:
# for a full list of available blocks (nn.Modules) in NeuroMANCER see:
print(blocks.blocks)
# for a full list of available activations in NeuroMANCER see:
print(activations.activations)

{'mlp': <class 'neuromancer.modules.blocks.MLP'>, 'mlp_dropout': <class 'neuromancer.modules.blocks.MLPDropout'>, 'mlp_bounds': <class 'neuromancer.modules.blocks.MLP_bounds'>, 'rnn': <class 'neuromancer.modules.blocks.RNN'>, 'pytorch_rnn': <class 'neuromancer.modules.blocks.PytorchRNN'>, 'linear': <class 'neuromancer.modules.blocks.Linear'>, 'residual_mlp': <class 'neuromancer.modules.blocks.ResMLP'>, 'basislinear': <class 'neuromancer.modules.blocks.BasisLinear'>, 'poly2': <class 'neuromancer.modules.blocks.Poly2'>, 'bilinear': <class 'neuromancer.modules.blocks.BilinearTorch'>, 'icnn': <class 'neuromancer.modules.blocks.InputConvexNN'>, 'pos_def': <class 'neuromancer.modules.blocks.PosDef'>}
{'softexp': <class 'neuromancer.modules.activations.SoftExponential'>, 'blu': <class 'neuromancer.modules.activations.BLU'>, 'aplu': <class 'neuromancer.modules.activations.APLU'>, 'prelu': <class 'neuromancer.modules.activations.PReLU'>, 'pelu': <class 'neuromancer.modules.activations.PELU'>, '

In [10]:
# 1, instantiate 4-layer multilayer perceptron with linear weight and ReLU activation
block_1 = blocks.MLP(insize=2, outsize=3,
                  bias=True,
                  linear_map=slim.maps['linear'],
                  nonlin=torch.nn.ReLU,
                  hsizes=[80] * 4)
# wrap modules into Node
node_4 = Node(block_1, ['x3'], ['y3'])
# evaluate forward pass of the node with dictionary input dataset
data = {'x3': torch.rand(10, 2)}
print(node_4(data).keys())
print(node_4(data)['y3'].shape)

dict_keys(['y3'])
torch.Size([10, 3])


In [11]:
# 2, instantiate recurrent neural net without bias, SVD linear map, and BLU activation
block_2 = blocks.RNN(insize=2, outsize=2,
                  bias=False,
                  linear_map=slim.linear.SVDLinear,
                  nonlin=activations.BLU,
                  hsizes=[80] * 4)
# wrap modules into Node
node_5 = Node(block_2, ['x4'], ['y4'])
# evaluate forward pass of the node with dictionary input dataset
data = {'x4': torch.rand(10, 2)}
print(node_5(data).keys())
print(node_5(data)['y4'].shape)

dict_keys(['y4'])
torch.Size([10, 2])


## System

**System** is a class that supports construction of cyclic computational graphs in NeuroMANCER.
System's graph is defined by a list of Nodes. Instantiated System can be used to simulate
dynamical systems in open or closed loop rollouts by specifying number of steps via nsteps.

In [18]:
# 1, create acyclic symbolic graph
# list of nodes to construct the graph
nodes = [node_1, node_2, node_3]
# n steps rollout
nsteps = 3
# connecting nodes via System class
system_1 = System(nodes, nsteps=nsteps)
# print input and output keys
print(system_1.input_keys)
print(system_1.output_keys)
# evaluate forward pass of the System with 3D input dataset
batch = 2
print(system_1({'x1': torch.rand(batch, nsteps, 1),
                'x2': torch.rand(batch, nsteps, 2)}))

['x1', 'x2']
['y3', 'y2', 'y1']
{'x1': tensor([[[0.4219],
         [0.6835],
         [0.9605]],

        [[0.3909],
         [0.6048],
         [0.4535]]]), 'x2': tensor([[[0.0882, 0.4910],
         [0.4187, 0.2001],
         [0.5172, 0.2064]],

        [[0.5336, 0.1138],
         [0.6999, 0.2038],
         [0.1567, 0.3752]]]), 'y1': tensor([[[-0.4077],
         [-0.2203],
         [-0.0220]],

        [[-0.4298],
         [-0.2767],
         [-0.3850]]], grad_fn=<CatBackward0>), 'y2': tensor([[[-0.3127],
         [-0.2817],
         [-0.2736]],

        [[-0.2732],
         [-0.2680],
         [-0.3065]]], grad_fn=<CatBackward0>), 'y3': tensor([[[-0.9131],
         [-0.5200],
         [-0.1188]],

        [[-0.9343],
         [-0.6252],
         [-0.8639]]], grad_fn=<CatBackward0>)}


In [19]:
# visualize symbolic computational graph
system_1.show()

FileNotFoundError: [Errno 2] "dot" not found in path.

In [20]:
nodes[2].output_keys

['y3']

In [21]:
# 2, close the loop by creating recursion in one of the nodes
nodes[2].output_keys = ['y1']
# create new system with cyclic computational graph
system_2 = System(nodes, nsteps=nsteps)
# print input and output keys
print(system_2.input_keys)
print(system_2.output_keys)
# evaluate forward pass of the System with 3D input dataset
print(system_1({'x1': torch.rand(batch, nsteps, 1),
                'x2': torch.rand(batch, nsteps, 2)}))

['x1', 'x2']
['y2', 'y1']
{'x1': tensor([[[0.5860],
         [0.5342],
         [0.0456]],

        [[0.0322],
         [0.7404],
         [0.6239]]]), 'x2': tensor([[[0.8341, 0.3792],
         [0.7088, 0.9969],
         [0.2293, 0.7143]],

        [[0.7183, 0.3604],
         [0.0671, 0.9612],
         [0.4712, 0.8910]]]), 'y1': tensor([[[-0.2901],
         [-0.6498],
         [-0.3272],
         [-1.3718],
         [-0.6771],
         [-0.7432]],

        [[-0.6866],
         [-1.4447],
         [-0.1796],
         [-2.9872],
         [-0.2630],
         [-0.4344]]], grad_fn=<CatBackward0>), 'y2': tensor([[[-0.2637],
         [-0.2687],
         [-0.2980]],

        [[-0.2673],
         [-0.3128],
         [-0.2743]]], grad_fn=<CatBackward0>)}


In [22]:
# visualize symbolic computational graph
system_2.show()

FileNotFoundError: [Errno 2] "dot" not found in path.